In [ ]:
# ============================================================
# BLOQUE CARGA RAPIDA: cargar YOLO sin re-entrenar
# (correr despues del Bloque 1, en lugar del entrenamiento)
# ============================================================
!pip install ultralytics -q
from ultralytics import YOLO
from google.colab import drive

# Montar Drive y cargar el modelo guardado
drive.mount('/content/drive')
rutaModeloYolo = "/content/drive/MyDrive/TrabajoFinal_PI/modelo_platanos_yolo.pt"
modeloYOLO = YOLO(rutaModeloYolo)
print("Modelo YOLO cargado desde Drive (sin re-entrenar).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive
Modelo YOLO cargado desde Drive (sin re-entrenar).


In [ ]:
# Definir las variables que la GUI necesita (por si no corriste el Bloque 1 en esta sesion)
nombresEspanol = {"unripe": "Verde", "ripe": "Maduro",
                  "overripe": "Sobremaduro", "rotten": "Podrido"}
clases = ["unripe", "ripe", "overripe", "rotten"]
print("Variables definidas:", nombresEspanol)

Variables definidas: {'unripe': 'Verde', 'ripe': 'Maduro', 'overripe': 'Sobremaduro', 'rotten': 'Podrido'}


In [ ]:
# ============================================================
# BLOQUE GUI YOLO (corregido: sin emoji, con nombre del modelo)
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from PIL import Image
import io
import matplotlib.pyplot as plt

nombresEspanol = {"unripe": "Verde", "ripe": "Maduro",
                  "overripe": "Sobremaduro", "rotten": "Podrido"}
clases = ["unripe", "ripe", "overripe", "rotten"]

infoLogistica = {
    "unripe": {"apto": "No apto para consumo inmediato (requiere maduracion)",
               "ventana": "7-14 dias", "mercado": "Mercados lejanos / Exportacion",
               "obs": "Resistente al transporte. Puede madurar en destino."},
    "ripe": {"apto": "Apto para consumo inmediato",
             "ventana": "3-7 dias", "mercado": "Mercados regionales / Consumo directo",
             "obs": "Estado optimo para venta al consumidor."},
    "overripe": {"apto": "Apto, preferible para procesamiento industrial",
                 "ventana": "1-2 dias", "mercado": "Mercado local / Industria (pures, harina)",
                 "obs": "Alto riesgo de perdida. Comercializar con urgencia."},
    "rotten": {"apto": "NO apto para consumo",
               "ventana": "0 dias (desecho)", "mercado": "Ninguno",
               "obs": "Separar del lote para evitar contaminacion."}
}

titulo = widgets.HTML(
    "<h2 style='color:#333'>🍌Clasificador de Madurez de Platanos</h2>"
    "<p style='color:#555'>Sube una foto de un platano y presiona Clasificar para "
    "obtener su estado de madurez y recomendacion logistica.</p>"
)
cargador = widgets.FileUpload(accept="image/*", multiple=False)
boton = widgets.Button(description="Clasificar", button_style="success",
                       icon="check", layout=widgets.Layout(width="200px", height="40px"))
salida = widgets.Output()

def procesar(b):
    with salida:
        clear_output()
        if len(cargador.value) == 0:
            print("Primero sube una imagen, luego presiona Clasificar.")
            return
        try:
            valor = cargador.value
            contenido = (list(valor.values())[0]["content"] if isinstance(valor, dict)
                         else valor[0]["content"])
        except Exception as e:
            print(f"Error leyendo el archivo: {e}")
            return

        imgPil = Image.open(io.BytesIO(contenido)).convert("RGB")
        imgArray = np.array(imgPil)

        resultado = modeloYOLO.predict(imgArray, verbose=False)
        probs = resultado[0].probs
        idxClase = probs.top1
        claseIngles = resultado[0].names[idxClase]
        claseEspanol = nombresEspanol[claseIngles]
        confianza = probs.top1conf.item() * 100
        info = infoLogistica[claseIngles]

        plt.figure(figsize=(5, 5))
        plt.imshow(imgPil); plt.axis("off")
        plt.title(f"Modelo YOLO11: {claseEspanol} ({confianza:.1f}%)", fontsize=13)
        plt.show()

        print("=" * 52)
        print(f"  MODELO YOLO11 -> {claseEspanol}  (confianza {confianza:.1f}%)")
        print("=" * 52)
        print(f"\n  ¿Apto para consumo?  {info['apto']}")
        print(f"  Ventana de transporte: {info['ventana']}")
        print(f"  Mercado recomendado:   {info['mercado']}")
        print(f"  Observacion:           {info['obs']}")
        print(f"\n  Probabilidades por clase:")
        todasProbs = probs.data.tolist()
        for i, nombreYolo in resultado[0].names.items():
            barra = "█" * int(todasProbs[i] * 25)
            print(f"    {nombresEspanol[nombreYolo]:14s}: {todasProbs[i]*100:5.1f}%  {barra}")

boton.on_click(procesar)
display(titulo, cargador, boton, salida)

HTML(value="<h2 style='color:#333'>🍌Clasificador de Madurez de Platanos</h2><p style='color:#555'>Sube una fot…

FileUpload(value={}, accept='image/*', description='Upload')

Button(button_style='success', description='Clasificar', icon='check', layout=Layout(height='40px', width='200…

Output()

In [ ]:
# ============================================================
# BLOQUE GUI CNN (corregido: sin emoji, con nombre del modelo)
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from PIL import Image
import io
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from google.colab import drive

# Cargar CNN desde Drive (si no esta cargado)
drive.mount('/content/drive')
modeloCNN = load_model("/content/drive/MyDrive/TrabajoFinal_PI/modelo_platanos_cnn.keras")
print("Modelo CNN cargado.\n")

nombresEspanol = {"unripe": "Verde", "ripe": "Maduro",
                  "overripe": "Sobremaduro", "rotten": "Podrido"}
clases = ["unripe", "ripe", "overripe", "rotten"]

infoLogistica = {
    "unripe": {"apto": "No apto para consumo inmediato (requiere maduracion)",
               "ventana": "7-14 dias", "mercado": "Mercados lejanos / Exportacion",
               "obs": "Resistente al transporte. Puede madurar en destino."},
    "ripe": {"apto": "Apto para consumo inmediato",
             "ventana": "3-7 dias", "mercado": "Mercados regionales / Consumo directo",
             "obs": "Estado optimo para venta al consumidor."},
    "overripe": {"apto": "Apto, preferible para procesamiento industrial",
                 "ventana": "1-2 dias", "mercado": "Mercado local / Industria (pures, harina)",
                 "obs": "Alto riesgo de perdida. Comercializar con urgencia."},
    "rotten": {"apto": "NO apto para consumo",
               "ventana": "0 dias (desecho)", "mercado": "Ninguno",
               "obs": "Separar del lote para evitar contaminacion."}
}

titulo = widgets.HTML(
    "<h2 style='color:#333'>🍌 Clasificador de Madurez de Platanos</h2>"
    "<p style='color:#555'>Sube una foto de un platano y presiona Clasificar para "
    "obtener su estado de madurez y recomendacion logistica.</p>"
)
cargador = widgets.FileUpload(accept="image/*", multiple=False)
boton = widgets.Button(description="Clasificar", button_style="info",
                       icon="check", layout=widgets.Layout(width="200px", height="40px"))
salida = widgets.Output()

def procesarCNN(b):
    with salida:
        clear_output()
        if len(cargador.value) == 0:
            print("Primero sube una imagen, luego presiona Clasificar.")
            return
        try:
            valor = cargador.value
            contenido = (list(valor.values())[0]["content"] if isinstance(valor, dict)
                         else valor[0]["content"])
        except Exception as e:
            print(f"Error leyendo el archivo: {e}")
            return

        imgPil = Image.open(io.BytesIO(contenido)).convert("RGB")
        imgArray = np.array(imgPil)
        img = cv2.resize(imgArray, (224, 224)).astype("float32") / 255.0
        img = np.expand_dims(img, axis=0)

        probs = modeloCNN.predict(img, verbose=0)[0]
        idxClase = np.argmax(probs)
        claseIngles = clases[idxClase]
        claseEspanol = nombresEspanol[claseIngles]
        confianza = probs[idxClase] * 100
        info = infoLogistica[claseIngles]

        plt.figure(figsize=(5, 5))
        plt.imshow(imgPil); plt.axis("off")
        plt.title(f"Modelo CNN (MobileNetV2): {claseEspanol} ({confianza:.1f}%)", fontsize=13)
        plt.show()

        print("=" * 52)
        print(f"  MODELO CNN (MobileNetV2) -> {claseEspanol}  (confianza {confianza:.1f}%)")
        print("=" * 52)
        print(f"\n  ¿Apto para consumo?  {info['apto']}")
        print(f"  Ventana de transporte: {info['ventana']}")
        print(f"  Mercado recomendado:   {info['mercado']}")
        print(f"  Observacion:           {info['obs']}")
        print(f"\n  Probabilidades por clase:")
        for i, c in enumerate(clases):
            barra = "█" * int(probs[i] * 25)
            print(f"    {nombresEspanol[c]:14s}: {probs[i]*100:5.1f}%  {barra}")

boton.on_click(procesarCNN)
display(titulo, cargador, boton, salida)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelo CNN cargado.



HTML(value="<h2 style='color:#333'>🍌 Clasificador de Madurez de Platanos</h2><p style='color:#555'>Sube una fo…

FileUpload(value={}, accept='image/*', description='Upload')

Button(button_style='info', description='Clasificar', icon='check', layout=Layout(height='40px', width='200px'…

Output()